# Week 7 — Hypothesis testing (simulation-first)

### Deliverable
- One completed test with effect size + CI + p-value + assumptions


In [1]:
# Core imports (kept minimal for beginners)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# Dataset URL (UCI Heart Disease - Cleveland)
UCI_URL = "https://www.archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

# Column names for processed.cleveland.data (14 columns commonly used in teaching)
COLS = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
        "exang","oldpeak","slope","ca","thal","num"]


In [2]:
from scipy import stats


In [3]:
import ssl
import io
import urllib.request # Added this import

def load_raw():
    # Create an unverified SSL context to bypass certificate verification
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    # Open the URL with the unverified context
    with urllib.request.urlopen(UCI_URL, context=ctx) as url_response:
        # Read the content and decode it
        s = url_response.read().decode('utf-8')

    # Use io.StringIO to make the string behave like a file for pandas.read_csv
    df_raw = pd.read_csv(io.StringIO(s), header=None, names=COLS)
    return df_raw

def coerce_types(df_raw):
    # Missing values are sometimes encoded as "?"
    df = df_raw.replace("?", np.nan).copy()

    # Convert numeric-looking columns
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Binary target: disease present if num > 0 (UCI convention)
    df["disease"] = (df["num"] > 0).astype("Int64")
    return df

df_raw = load_raw()
df = coerce_types(df_raw)

df.head()


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,disease
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,0


In [4]:
def clean_heart(df_raw):
    df = df_raw.replace("?", np.nan).copy()
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["disease"] = (df["num"] > 0).astype(int)
    return df.dropna(subset=["disease"])

df_clean = clean_heart(df_raw)


## Helper functions: permutation test + bootstrap CI


In [5]:
rng = np.random.default_rng(0)

def permutation_test_diff_means(a, b, n_perm=5000):
    a = np.asarray(a); b = np.asarray(b)
    a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
    obs = a.mean() - b.mean()
    pooled = np.concatenate([a, b])
    n_a = len(a)
    more_extreme = 0
    for _ in range(n_perm):
        rng.shuffle(pooled)
        diff = pooled[:n_a].mean() - pooled[n_a:].mean()
        if abs(diff) >= abs(obs):
            more_extreme += 1
    p = (more_extreme + 1) / (n_perm + 1)
    return obs, p

def bootstrap_ci_diff_means(a, b, n_boot=5000, alpha=0.05):
    a = np.asarray(a); b = np.asarray(b)
    a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
    diffs = []
    for _ in range(n_boot):
        diffs.append(rng.choice(a, size=len(a), replace=True).mean()
                     - rng.choice(b, size=len(b), replace=True).mean())
    lo, hi = np.quantile(diffs, [alpha/2, 1-alpha/2])
    return (lo, hi)

def cohens_d(a, b):
    a = np.asarray(a); b = np.asarray(b)
    a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
    n1, n2 = len(a), len(b)
    s1, s2 = a.var(ddof=1), b.var(ddof=1)
    sp = ((n1-1)*s1 + (n2-1)*s2) / (n1+n2-2)
    return (a.mean() - b.mean()) / np.sqrt(sp)


## Task A — Numeric vs disease


In [6]:
col = "thalach"  # TODO
a = df_clean.loc[df_clean["disease"]==0, col].dropna()
b = df_clean.loc[df_clean["disease"]==1, col].dropna()

obs, p_perm = permutation_test_diff_means(a, b, n_perm=5000)
ci = bootstrap_ci_diff_means(a, b, n_boot=5000)
d = cohens_d(a, b)

print("Variable:", col)
print("Mean difference (no disease - disease):", obs)
print("Permutation p-value:", p_perm)
print("Bootstrap 95% CI:", ci)
print("Cohen's d:", d)


Variable: thalach
Mean difference (no disease - disease): 19.11905597473242
Permutation p-value: 0.0001999600079984003
Bootstrap 95% CI: (np.float64(14.486677487278472), np.float64(23.874803693630454))
Cohen's d: 0.9181263397445941


## Task B — Categorical vs disease (chi-square)


In [7]:
cat = "cp"  # TODO
ct = pd.crosstab(df_clean[cat], df_clean["disease"])
display(ct)

chi2, p, dof, expected = stats.chi2_contingency(ct)
print("chi2:", chi2, "p:", p)

# Cramér's V
n = ct.to_numpy().sum()
r, k = ct.shape
cramers_v = np.sqrt((chi2 / n) / (min(r-1, k-1)))
print("Cramér's V:", cramers_v)


disease,0,1
cp,,
1.0,16,7
2.0,41,9
3.0,68,18
4.0,39,105


chi2: 81.81577027653815 p: 1.2517106007837527e-17
Cramér's V: 0.5196335668689597


## TODO — Assumptions & interpretation

Permutation test: thalach by disease group
- Assumption 1: The permutation test assumes that under H0, the disease labels are exchangeable -
i.e., the two groups (disease / no disease) are drawn from the same distribution.
This holds if patients were randomly assigned, but here they were not - this is
observational data. Systematic differences in age, sex, or other factors could
violate this assumption.
- Assumption 2: Each patient is assumed to be an independent observation. This is reasonable for
this dataset (one row = one patient, no repeated measures), but patients from the
same clinic may share demographic or lifestyle characteristics that introduce
subtle clustering.
- Interpretation: Patients without heart disease achieve a significantly higher maximum heart rate
(thalach) than patients with disease (mean difference ≈ +19 bpm, permutation
p < 0.001). The bootstrap 95% CI does not include zero, confirming the difference
is statistically reliable. Cohen's d ≈ 1.0 indicates a large effect size - thalach
is one of the strongest numerical separators between the two groups in this dataset.
- Limitation: This is a correlation, not causation. Lower thalach in disease patients may reflect
reduced cardiac capacity due to disease, or it may partly be confounded by age
(older patients have lower max HR and higher disease rates). A multivariate model
would be needed to isolate thalach's independent contribution.


Chi-square test: chest pain type (cp) vs disease
- Assumption 1 (Expected cell counts ≥ 5):
The chi-square approximation is valid only when all expected cell frequencies are
≥ 5. If any cell is below 5, Fisher's exact test or
collapsing categories would be more appropriate

- Assumption 2 (Independence of observations):
Same as Task A - each patient must be an independent observation. No patient
should appear in more than one cp category (which is guaranteed here since cp
is a single recorded value per patient).

- There is a statistically significant association between chest pain type and
heart disease presence (chi-square p < 0.001). Cramér's V ≈ 0.49 indicates a
moderate-to-large association. Notably, patients classified as "asymptomatic"
(cp = 4) have the highest disease rate - a clinically important finding that
contradicts the naive expectation that symptomatic patients are sicker.

- Chi-square and Cramér's V only measure association - they do not tell us the
direction or ordering of the effect. A logistic regression with cp as a
categorical predictor would give a clearer picture of which specific chest pain
type drives the most risk.